# Практическая работа: логистическая регрессия и SVM

**Цель:** предсказать уровень дохода человека (`income`) по данным из `adult.csv` и сравнить качество двух моделей на тестовой выборке с помощью метода `.score()`.

Файл `adult.csv` должен находиться в той же папке, что и этот ноутбук.

## 1. Импорт библиотек

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

## 2. Загрузка данных и знакомство с ними

In [2]:
import numpy as np

data = pd.read_csv("adult.csv")

print("Размер таблицы:", data.shape)
display(data.head())
print("\\nТипы данных и количество непустых значений:")
data.info()

Размер таблицы: (48842, 15)


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


\nТипы данных и количество непустых значений:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


В целевом столбце `income` находится уровень дохода. Остальные столбцы используем как признаки.

В датасете символ `?` обозначает пропущенное значение. Заменим его на `NaN`, чтобы затем заполнить пропуски наиболее частым значением соответствующего признака.

In [3]:
data = data.replace("?", np.nan)

target = "income"
X = data.drop(columns=[target])
y = data[target]

print("Распределение целевой переменной:")
print(y.value_counts())

Распределение целевой переменной:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64


## 3. Разделение на обучающую и тестовую выборки

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

Обучающая выборка: (39073, 14)
Тестовая выборка: (9769, 14)


## 4. Предварительная обработка признаков

Модели машинного обучения должны получать числовые признаки. Поэтому:
- числовые признаки заполним медианой и стандартизируем;
- категориальные признаки заполним наиболее частым значением и преобразуем в числовые столбцы с помощью `OneHotEncoder`.

Предобработка включена в `Pipeline`, чтобы параметры преобразований рассчитывались только по обучающей выборке.

In [5]:
numeric_features = X.select_dtypes(include=["number"]).columns
categorical_features = X.select_dtypes(exclude=["number"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 5. Модель логистической регрессии

In [6]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

logistic_score = logistic_model.score(X_test, y_test)
print(f"Качество логистической регрессии на тестовой выборке: {logistic_score:.4f}")

Качество логистической регрессии на тестовой выборке: 0.8524


## 6. Модель SVM

Используем `LinearSVC` — вариант метода опорных векторов с линейной разделяющей поверхностью. Он подходит для этого датасета, поскольку содержит десятки тысяч строк.

In [7]:
svm_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(max_iter=5000, random_state=42))
])

svm_model.fit(X_train, y_train)

svm_score = svm_model.score(X_test, y_test)
print(f"Качество SVM на тестовой выборке: {svm_score:.4f}")

Качество SVM на тестовой выборке: 0.8530


## 7. Сравнение результатов

In [8]:
results = pd.DataFrame({
    "Модель": ["Логистическая регрессия", "SVM (LinearSVC)"],
    "Качество на тестовой выборке (.score)": [logistic_score, svm_score]
})

display(results.sort_values(
    by="Качество на тестовой выборке (.score)",
    ascending=False
).reset_index(drop=True))

,Модель,Качество на тестовой выборке (.score)
0,SVM (LinearSVC),0.853004
1,Логистическая регрессия,0.852390


## Вывод

Метод `.score()` для классификаторов возвращает **accuracy** — долю правильных ответов на тестовой выборке.

Ниже приведён итог по фактическим результатам запуска обеих моделей.


In [9]:
best_name = "логистическая регрессия" if logistic_score > svm_score else "SVM (LinearSVC)" if svm_score > logistic_score else "обе модели (одинаковое качество)"
print(f"Логистическая регрессия: {logistic_score:.4f} ({logistic_score:.2%})")
print(f"SVM (LinearSVC): {svm_score:.4f} ({svm_score:.2%})")
print(f"Лучший результат: {best_name}.")

Логистическая регрессия: 0.8524 (85.24%)
SVM (LinearSVC): 0.8530 (85.30%)
Лучший результат: SVM (LinearSVC).
